# Problem 1: CNN on MNIST (torchvision)

Same workflow as `1_pytorch_basic_cnn_digits.ipynb`, but instead of the tiny
8x8 Sklearn `load_digits` dataset we use the real MNIST dataset from
`torchvision` (28 x 28 grayscale digit images). We train a small CNN, evaluate
it on the MNIST test set, and finally run inference on a few images of
handwritten digits.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from torchvision import datasets, transforms

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
)

# Random processes are used when splitting and shuffling data and when
# initializing model weights. A fixed seed makes results more repeatable.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Apple Silicon use MPS; laptops with NVIDIA GPUs can use CUDA.
# MNIST is still small enough to train quickly on a CPU. So dont worry
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")


## Load the MNIST dataset (28 x 28) with `torchvision`

`torchvision.datasets.MNIST` already ships with a train/test split (60,000 /
10,000 images), so there's no need for `train_test_split` here. Each image is
1 x 28 x 28 (single grayscale channel). We normalize with MNIST's well-known
mean/std so pixel values are roughly zero-centered, which helps training.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                    # scales pixels to [0, 1], shape (1, 28, 28)
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST mean/std
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples:  {len(test_dataset)}")
print(f"Image shape:   {train_dataset[0][0].shape}")

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


## Peek at a few training images

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax in axes:
    idx = random.randrange(len(train_dataset))
    image, label = train_dataset[idx]
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.tight_layout()
plt.show()


## Define a simple CNN

Two conv/pool blocks bring the 28x28 input down to a 7x7 feature map, which
is then flattened and passed through a couple of fully-connected layers.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 28x28 -> 14x14
        x = self.pool(F.relu(self.conv2(x)))  # 14x14 -> 7x7
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleCNN().to(device)
print(model)


## Train

In [ ]:
EPOCHS = 5
LEARNING_RATE = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

model.train()
for epoch in range(1, EPOCHS + 1):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch {epoch}/{EPOCHS} - loss: {epoch_loss:.4f}")


## Evaluate on the MNIST test set

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

accuracy = accuracy_score(all_labels, all_preds)
print(f"Test accuracy: {accuracy:.4f}\n")
print(classification_report(all_labels, all_preds, digits=4))

ConfusionMatrixDisplay.from_predictions(all_labels, all_preds)
plt.show()


## Inference on your own handwriting

Drop a few photos of digits you've drawn by hand into the
`handwritten_digits/` folder (any image format, roughly square crops work
best) and re-run this cell. Each image is converted to grayscale, resized to
28 x 28, and inverted/normalized to match MNIST's white-digit-on-black-
background convention before being fed to the trained model.

If the folder is empty, a few simple placeholder "handwritten-style" digit
images are generated on the fly with `PIL.ImageDraw` so the inference cell
still has something to run on — swap these out for your own real
handwriting photos any time. Just a handful of images is enough here, per
the assignment ("do not overdo this").

In [ ]:
import os
from PIL import Image, ImageDraw

HANDWRITTEN_DIR = "handwritten_digits"
os.makedirs(HANDWRITTEN_DIR, exist_ok=True)


def _make_placeholder_digit(digit, path, size=200, seed=0):
    """Draw a simple, slightly jittered stroke-based digit as a stand-in
    for a real handwriting photo (black strokes on a white canvas)."""
    rng = random.Random(seed)
    img = Image.new("L", (size, size), color=255)
    draw = ImageDraw.Draw(img)
    jitter = lambda v: v + rng.randint(-6, 6)
    width = size // 10

    strokes = {
        0: [[(0.3, 0.2), (0.7, 0.2), (0.75, 0.5), (0.7, 0.8), (0.3, 0.8), (0.25, 0.5), (0.3, 0.2)]],
        1: [[(0.5, 0.15), (0.5, 0.85)], [(0.35, 0.3), (0.5, 0.15)]],
        3: [[(0.25, 0.2), (0.7, 0.2), (0.45, 0.48), (0.7, 0.55), (0.7, 0.8), (0.25, 0.8)]],
        7: [[(0.2, 0.2), (0.8, 0.2), (0.4, 0.85)]],
    }
    for stroke in strokes.get(digit, strokes[1]):
        pts = [(jitter(int(x * size)), jitter(int(y * size))) for x, y in stroke]
        draw.line(pts, fill=0, width=width, joint="curve")
    img.save(path)


existing = [f for f in os.listdir(HANDWRITTEN_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
if not existing:
    print("No images found in 'handwritten_digits/' -- generating a few placeholder samples.")
    placeholders = [(0, 1), (1, 2), (3, 3), (7, 4)]
    for digit, idx in placeholders:
        _make_placeholder_digit(digit, os.path.join(HANDWRITTEN_DIR, f"digit_{digit}_{idx}.png"), seed=idx)
    existing = [f for f in os.listdir(HANDWRITTEN_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg"))]

print(f"Running inference on {len(existing)} image(s): {existing}")


In [ ]:
def preprocess_handwritten(path):
    img = Image.open(path).convert("L").resize((28, 28), Image.LANCZOS)
    arr = np.array(img).astype(np.float32) / 255.0

    # If the drawing is dark strokes on a light background (typical of a
    # photo/scan), invert it so digits are white-on-black like MNIST.
    if arr.mean() > 0.5:
        arr = 1.0 - arr

    arr = (arr - 0.1307) / 0.3081
    tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).float()  # (1, 1, 28, 28)
    return tensor, arr


model.eval()
fig, axes = plt.subplots(1, len(existing), figsize=(3 * len(existing), 3))
if len(existing) == 1:
    axes = [axes]

for ax, fname in zip(axes, sorted(existing)):
    path = os.path.join(HANDWRITTEN_DIR, fname)
    tensor, display_arr = preprocess_handwritten(path)
    with torch.no_grad():
        logits = model(tensor.to(device))
        pred = logits.argmax(dim=1).item()

    ax.imshow(display_arr, cmap="gray")
    ax.set_title(f"{fname}\npredicted: {pred}")
    ax.axis("off")

plt.tight_layout()
plt.show()
